In [1]:
# ================================
# STEP 1: Install Gradio
# ================================
# !pip install gradio


# ================================
# STEP 2: Dummy Database (Tool Layer)
# ================================
customers = {
    "C101": {
        "name": "Amit Sharma",
        "balance": "₹85,000",
        "transactions": [
            "₹2000 - Amazon",
            "₹5000 - ATM Withdrawal",
            "₹1200 - Swiggy",
            "₹15,000 - Salary Credit",
            "₹3000 - Electricity Bill"
        ],
        "emi_due": "5 April 2026"
    },

    "C102": {
        "name": "Neha Verma",
        "balance": "₹1,25,500",
        "transactions": [
            "₹2500 - Flipkart",
            "₹800 - Uber",
            "₹2000 - Grocery",
            "₹20,000 - Salary Credit",
            "₹4000 - Rent"
        ],
        "emi_due": "10 April 2026"
    }
}


# ================================
# STEP 3: Tool Functions (MCP Tools)
# ================================
def get_balance(customer_id):
    if customer_id in customers:
        return f"💰 Balance: {customers[customer_id]['balance']}"
    return "❌ Customer not found"


def get_transactions(customer_id):
    if customer_id in customers:
        txns = "\n".join(customers[customer_id]['transactions'])
        return f"📜 Last 5 Transactions:\n{txns}"
    return "❌ Customer not found"


def get_emi(customer_id):
    if customer_id in customers:
        return f"📅 EMI Due Date: {customers[customer_id]['emi_due']}"
    return "❌ Customer not found"


# ================================
# STEP 4: Security Layer
# ================================
def secure_access(user_id, requested_id):
    return user_id == requested_id


# ================================
# STEP 5: MCP Agent Logic
# ================================
def banking_agent(message, customer_id, history):

    # simulate logged-in user
    user_id = customer_id

    # security check
    if not secure_access(user_id, customer_id):
        history.append({
            "role": "assistant",
            "content": "🚫 Access Denied"
        })
        return "", history

    message_lower = message.lower()

    # ================================
    # Tool Invocation Logic
    # ================================
    if "balance" in message_lower:
        response = get_balance(customer_id)

    elif "transaction" in message_lower:
        response = get_transactions(customer_id)

    elif "emi" in message_lower or "loan" in message_lower:
        response = get_emi(customer_id)

    elif "hi" in message_lower or "hello" in message_lower:
        response = "👋 Hello! You can ask about balance, transactions, or loan EMI."

    else:
        response = "🤖 I can help with balance, transactions, or EMI details."

    # ================================
    # Store Chat History
    # ================================
    history.append({
        "role": "user",
        "content": message
    })

    history.append({
        "role": "assistant",
        "content": response
    })

    return "", history


# ================================
# STEP 6: Gradio Chat UI
# ================================
import gradio as gr

with gr.Blocks() as demo:

    gr.Markdown("# 🏦 Banking Smart Assistant")

    customer_id = gr.Textbox(
        label="Enter Customer ID (example: C101)"
    )

    chatbot = gr.Chatbot()

    msg = gr.Textbox(
        label="Ask your banking question"
    )

    state = gr.State([])

    msg.submit(
        banking_agent,
        inputs=[msg, customer_id, state],
        outputs=[msg, chatbot]
    )

demo.launch(share=True)

C:\Users\sk165\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


* Running on local URL:  http://127.0.0.1:7862

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.


In [2]:
# ================================
# STEP 1: Imports + LLM Init
# ================================
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage

model = init_chat_model(
    model="mistral-medium"   # or any available model
)


# ================================
# STEP 2: Dummy Database
# ================================
customers = {
    "C101": {
        "name": "Amit Sharma",
        "balance": "₹85,000",
        "transactions": [
            "₹2000 - Amazon",
            "₹5000 - ATM Withdrawal",
            "₹1200 - Swiggy",
            "₹15,000 - Salary Credit",
            "₹3000 - Electricity Bill"
        ],
        "emi_due": "5 April 2026"
    }
}


# ================================
# STEP 3: MCP Tools
# ================================
def get_balance(customer_id):
    return f"💰 Balance: {customers[customer_id]['balance']}"


def get_transactions(customer_id):
    txns = "\n".join(customers[customer_id]['transactions'])
    return f"📜 Last 5 Transactions:\n{txns}"


def get_emi(customer_id):
    return f"📅 EMI Due Date: {customers[customer_id]['emi_due']}"


# ================================
# STEP 4: Tool Registry (MCP Style)
# ================================
TOOLS = {
    "balance": get_balance,
    "transactions": get_transactions,
    "emi": get_emi
}


# ================================
# STEP 5: LLM Intent Classifier
# ================================
def detect_intent(user_query):

    prompt = f"""
    You are a banking assistant.

    Classify the user query into ONE of these intents:
    - balance
    - transactions
    - emi

    Only return the intent name.

    Query: {user_query}
    """

    response = model.invoke([HumanMessage(content=prompt)])

    return response.content.strip().lower()


# ================================
# STEP 6: Security Layer
# ================================
def secure_access(user_id, requested_id):
    return user_id == requested_id


# ================================
# STEP 7: MCP Agent (LLM + Tools)
# ================================
def banking_agent(message, customer_id, history):

    user_id = customer_id

    # security check
    if not secure_access(user_id, customer_id):
        history.append({"role": "assistant", "content": "🚫 Access Denied"})
        return "", history

    # ================================
    # LLM decides intent
    # ================================
    intent = detect_intent(message)

    # ================================
    # Tool Execution
    # ================================
    if intent in TOOLS:
        response = TOOLS[intent](customer_id)
    else:
        response = "🤖 Sorry, I couldn't understand your request."

    # ================================
    # Save Chat
    # ================================
    history.append({"role": "user", "content": message})
    history.append({"role": "assistant", "content": response})

    return "", history


# ================================
# STEP 8: Gradio UI
# ================================
import gradio as gr

with gr.Blocks() as demo:

    gr.Markdown("# 🏦 Banking MCP + LLM Assistant")

    customer_id = gr.Textbox(label="Customer ID (C101)")

    chatbot = gr.Chatbot()

    msg = gr.Textbox(label="Ask your banking question")

    state = gr.State([])

    msg.submit(
        banking_agent,
        inputs=[msg, customer_id, state],
        outputs=[msg, chatbot]
    )

demo.launch(share=True)

* Running on local URL:  http://127.0.0.1:7863

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.


In [4]:
# ================================
# STEP 1: Imports + Mistral LLM
# ================================
from langchain_mistralai import ChatMistralAI
from langchain_core.messages import HumanMessage
import os
from dotenv import load_dotenv
load_dotenv()
api_key = os.environ["MISTRAL_API_KEY"] 

model = ChatMistralAI(
    model="mistral-large-latest",   # or mistral-medium
    temperature=0,
    api_key=api_key
)


# ================================
# STEP 2: Users + Roles (RBAC DB)
# ================================
users = {
    "C101": {"name": "Amit Sharma", "role": "customer"},
    "C102": {"name": "Neha Verma", "role": "customer"},
    "ADMIN1": {"name": "Bank Manager", "role": "admin"}
}


# ================================
# STEP 3: Banking Data
# ================================
customers = {
    "C101": {
        "balance": "₹85,000",
        "transactions": [
            "₹2000 - Amazon",
            "₹5000 - ATM Withdrawal",
            "₹1200 - Swiggy",
            "₹15,000 - Salary Credit",
            "₹3000 - Electricity Bill"
        ],
        "emi_due": "5 April 2026"
    },

    "C102": {
        "balance": "₹1,25,500",
        "transactions": [
            "₹2500 - Flipkart",
            "₹800 - Uber",
            "₹2000 - Grocery",
            "₹20,000 - Salary Credit",
            "₹4000 - Rent"
        ],
        "emi_due": "10 April 2026"
    }
}


# ================================
# STEP 4: MCP Tools
# ================================
def get_balance(customer_id):
    return f"💰 Balance: {customers[customer_id]['balance']}"


def get_transactions(customer_id):
    txns = "\n".join(customers[customer_id]['transactions'])
    return f"📜 Last 5 Transactions:\n{txns}"


def get_emi(customer_id):
    return f"📅 EMI Due Date: {customers[customer_id]['emi_due']}"


TOOLS = {
    "balance": get_balance,
    "transactions": get_transactions,
    "emi": get_emi
}


# ================================
# STEP 5: LLM Intent Detection (with safety)
# ================================
def detect_intent(user_query):

    prompt = f"""
    Classify banking query into:
    balance, transactions, emi

    Return only one word.

    Query: {user_query}
    """

    try:
        response = model.invoke([HumanMessage(content=prompt)])
        return response.content.strip().lower()

    except Exception as e:
        print("LLM Error:", e)
        return "unknown"


# ================================
# STEP 6: RBAC Authorization
# ================================
def authorize(user_id, target_customer_id):

    if user_id not in users:
        return False, "❌ Invalid User"

    role = users[user_id]["role"]

    if role == "admin":
        return True, "✅ Admin Access Granted"

    if role == "customer" and user_id == target_customer_id:
        return True, "✅ Access Granted"

    return False, "🚫 Unauthorized Access"


# ================================
# STEP 7: MCP Agent
# ================================
def banking_agent(message, user_id, target_customer_id, history):

    # Authorization
    is_allowed, auth_msg = authorize(user_id, target_customer_id)

    if not is_allowed:
        history.append({"role": "assistant", "content": auth_msg})
        return "", history

    # Intent detection via Mistral
    intent = detect_intent(message)

    # Tool execution
    if intent in TOOLS:
        response = TOOLS[intent](target_customer_id)
    else:
        response = "⚠️ Couldn't process request. Try asking about balance, transactions, or EMI."

    # Save chat
    history.append({"role": "user", "content": message})
    history.append({"role": "assistant", "content": response})

    return "", history


# ================================
# STEP 8: Gradio UI
# ================================
import gradio as gr

with gr.Blocks() as demo:

    gr.Markdown("# 🏦 Banking Assistant (MistralAI + MCP + RBAC)")

    user_id = gr.Textbox(label="Login User ID (C101 / ADMIN1)")
    target_customer_id = gr.Textbox(label="Customer ID to Access (C101 / C102)")

    chatbot = gr.Chatbot()

    msg = gr.Textbox(label="Ask your banking question")

    state = gr.State([])

    msg.submit(
        banking_agent,
        inputs=[msg, user_id, target_customer_id, state],
        outputs=[msg, chatbot]
    )

demo.launch(share=True)

* Running on local URL:  http://127.0.0.1:7865

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.
